# EN.520.681, Perception for Embodied AI, Spring 2026
# Project 1
#### Due: 3/9 11:59 ET
submission instructions: 

Please submit these files to gradescope, do not change their name or the autograder might not find them properly.
* gaussians.py
* render.py
* val_[0-180].png

In [ ]:
# ========== Colab + Google Drive 专用：先运行此 cell ==========
# 若在本地运行，可跳过此 cell
# 前提：Drive 里已有 gaussians.py, render.py, camera.py, utils.py 和 解压后的 lego/ 文件夹
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/PoEAI_HW2'  # 改成你的路径（含 py 文件和 lego 文件夹的目录）
os.chdir(PROJECT_PATH)
!ls -la
!ls lego/

In [ ]:
# Install the following packages
!pip install plyfile
!pip install torchmetrics

In [ ]:
import os
import torch
import numpy as np

from PIL import Image
from tqdm import tqdm
from gaussians import Gaussians
from render import Renderer
from torch.utils.data import DataLoader, Dataset
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torchmetrics.functional import structural_similarity_index_measure
import matplotlib.pyplot as plt

In [ ]:
# If you are running on Colab, upload the data file to workspace and run this cell
import zipfile
def prepare_lego_dataset(base_dir="."):
    zip_path = os.path.join(base_dir, "lego.zip")
    extract_dir = os.path.join(base_dir, "lego")

    # Check if the zip file exists AND the folder does not exist
    if os.path.exists(zip_path) and not os.path.exists(extract_dir):
        print(f"Found '{zip_path}'. Extracting...")
        
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(base_dir)
            
        print("Extraction complete!")
    elif os.path.exists(extract_dir):
        print(f"Directory '{extract_dir}' already exists. Skipping unzip.")
    else:
        print(f"Error: '{zip_path}' not found in the directory.")

prepare_lego_dataset('./')

In [ ]:
def setup_optimizer(gaussians):

    gaussians.check_if_trainable()
    # HINT: Modify the learning rates to reasonable values. We have intentionally
    # set very high learning rates for all parameters. You can refer to open source code of the original paper.
    parameters = [
        {'params': [gaussians.opacities], 'lr': 0.05, "name": "opacities"},
        {'params': [gaussians.scales], 'lr': 0.005, "name": "scales"},
        {'params': [gaussians.colors], 'lr': 0.0025, "name": "colors"},
        {'params': [gaussians.means], 'lr': 0.00016, "name": "means"},
        {'params': [gaussians.quats], 'lr': 0.001, "name": "quats"}
    ]
    optimizer = torch.optim.Adam(parameters, lr=0.0, eps=1e-15)

    return optimizer

In [ ]:
def calculate_loss(pred_img, gt_img, lamb):
    '''Calculate the loss function. You can use structural_similarity_index_measure from torchmetrics 
    and l1_loss from torch.nn.functional. See eq (11)
    
    Args:
        pred_img    :   torch.tensor, rendered image in (H, W, C)
        gt_img      :   torch.tensor, gt image in (H, W, C)
        lamb        :   coefficient lambda (float) that balance l1_loss and ssim_loss
    '''

    pred_img = pred_img.permute(2,0,1).unsqueeze(0) # (C, H, W)
    gt_img = gt_img.permute(2,0,1).unsqueeze(0) # (C, H, W)
    
    ### YOUR CODE STARTS HERE ###
    l1_loss = torch.nn.functional.l1_loss(pred_img, gt_img)
    ssim_loss = 1.0 - structural_similarity_index_measure(pred_img, gt_img, data_range=1.0)
    total_loss = (1 - lamb) * l1_loss + lamb * ssim_loss
    ### YOUR CODE ENDS HERE ###

    return total_loss, l1_loss.detach().item(), ssim_loss.detach().item()



In [ ]:

root_path = './lego'
from utils import GSDataset
train_dataset = GSDataset(root=root_path, split='train', image_size=100, ply_path=root_path + '/sparse_model.ply')
test_dataset = GSDataset(root=root_path, split='test', image_size=100, ply_path=root_path + '/sparse_model.ply')
val_dataset = GSDataset(root=root_path, split='val', image_size=100, ply_path=root_path + '/sparse_model.ply')


train_loader = DataLoader(
    train_dataset, batch_size=1, shuffle=True, num_workers=0,
    drop_last=True, collate_fn=GSDataset.collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=1, shuffle=False, num_workers=0,
    drop_last=True, collate_fn=GSDataset.collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=1, shuffle=False, num_workers=0,
    drop_last=True, collate_fn=GSDataset.collate_fn
)
train_itr = iter(train_loader)
val_list = list(val_loader)

fixed_test_data = val_list[2]


In [ ]:
# Training parameters
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
num_itrs = 3000
viz_freq = 100
lamb = 0.2

# Density Control Parameters
opacity_reset_iter = 1000
density_control = True
densify_until_iter = num_itrs - opacity_reset_iter
densify_iter_start = 1000
densify_iter = 100
max_densify_ratio = 0.01

# Initialization
# Feel free to explore using different initializations
from_points = True

if from_points:
    gaussians = Gaussians(
            load_path=train_dataset.points_path, init_type="points",
            device=device, isotropic=False
        )
else:
    gaussians = Gaussians(
            load_path=train_dataset.points_path, init_type="random",
            device=device, isotropic=False, num_points=10000
        )

# Making gaussians trainable and setting up optimizer
gaussians.train()
optimizer = setup_optimizer(gaussians)
renderer = Renderer(gaussians, optimizer, extent=1.0, percent_dense=0.02, grad_threshold=5e-6, device=device)

# Training loop
viz_frames = []
for itr in range(num_itrs):

    # Fetching data
    try:
        data = next(train_itr)
    except StopIteration:
        train_itr = iter(train_loader)
        data = next(train_itr)

    gt_img, camera, gt_mask = data


    gt_img = gt_img[0].to(device)
    camera = camera[0].cuda() if device == "cuda" else camera[0]
    if gt_mask is not None:
        gt_mask = gt_mask[0].to(device)


    pred_img, _, means_3D = renderer.render(camera=camera, per_splat=-1, img_size=(gt_img.shape[1], gt_img.shape[0]),bg_colour=(0,0,0))

    loss, l1, ssim = calculate_loss(pred_img, gt_img, lamb)
    loss.backward()
    if itr > densify_iter_start and itr < densify_until_iter:
        with torch.no_grad():
            # means_3D.grad will be populated after backward()
            renderer.accumulate_grads(means_3D.grad)

    renderer.optimizer.step()
    renderer.optimizer.zero_grad()

    with torch.no_grad():

        if itr < densify_until_iter:
            if itr > densify_iter_start and itr % densify_iter == 0 and density_control:
                renderer.prune_and_densify(min_opacity=0.05, max_densify_ratio=max_densify_ratio)

            # Trick: periodically reset opacity to 0.01 to prevent stucking in local minimum
            if itr >= opacity_reset_iter and itr % opacity_reset_iter == 0:
                renderer.gaussians.reset_opacity(renderer.optimizer)

        if itr % viz_freq == 0:
        
            gt_img, camera, gt_mask = fixed_test_data
            gt_img = gt_img[0].to(device)
            camera = camera[0].cuda() if device == "cuda" else camera[0]
            if gt_mask is not None:
                gt_mask = gt_mask[0].to(device)
            pred_img, _, _ = renderer.render(camera=camera, per_splat=-1, img_size=(gt_img.shape[1], gt_img.shape[0]),bg_colour=(0,0,0))

            loss, l1, ssim = calculate_loss(pred_img, gt_img, lamb)
            
            pred_vis = pred_img.detach().cpu().numpy()
            gt_vis = gt_img.detach().cpu().numpy()

            pred_vis = np.clip(pred_vis, 0.0, 1.0)
            gt_vis   = np.clip(gt_vis, 0.0, 1.0)

            # Absolute error map
            err_vis = np.abs(pred_vis - gt_vis)
            err_gray = err_vis.mean(axis=-1)  # (H, W)

            psnr = peak_signal_noise_ratio(gt_vis, pred_vis)

            print(f"[*] Itr: {itr:07d} | #Points: {renderer.gaussians.get_num_points()} Loss: {loss:0.3f} - l1 {l1} - ssim {ssim} | psnr: {psnr}")


            plt.figure(figsize=(12, 4))

            plt.subplot(1, 3, 1)
            plt.imshow(gt_vis)
            plt.title("Ground Truth")
            plt.axis("off")

            plt.subplot(1, 3, 2)
            plt.imshow(pred_vis)
            plt.title("Prediction")
            plt.axis("off")

            plt.subplot(1, 3, 3)
            plt.imshow(err_gray, cmap="inferno")
            plt.title("Absolute Error")
            plt.colorbar(fraction=0.046, pad=0.04)
            plt.axis("off")

            plt.tight_layout()
            plt.show()

print("[*] Training Completed.")

In [ ]:
# Run validation after training to visualize more views
psnr_vals, ssim_vals = [], []
for i, val_data in enumerate(tqdm(val_loader, desc="Running Evaluation")):

    gt_img, camera, gt_mask = val_data
    gt_img = gt_img[0].to(device)
    camera = camera[0].cuda() if device == "cuda" else camera[0]
    if gt_mask is not None:
        gt_mask = gt_mask[0].to(device)

    with torch.no_grad():

        if gt_mask is not None:
            gt_mask = gt_mask[0].to(device)
        pred_img, _, _ = renderer.render(camera=camera, per_splat=-1, img_size=(gt_img.shape[1], gt_img.shape[0]),bg_colour=(0,0,0))

        gt_npy = gt_img.detach().cpu().numpy()
        pred_npy = pred_img.detach().cpu().numpy()

        psnr = peak_signal_noise_ratio(gt_npy, pred_npy)
        ssim = structural_similarity(gt_npy, pred_npy, channel_axis=-1, data_range=1.0)

        psnr_vals.append(psnr)
        ssim_vals.append(ssim)
        if i%1 == 0:
            print(f"PSNR: {psnr:.3f}")

            plt.figure(figsize=(12, 4))

            plt.subplot(1, 2, 1)
            plt.imshow(gt_npy)
            plt.title("Ground Truth")
            plt.axis("off")

            plt.subplot(1, 2, 2)
            plt.imshow(pred_npy)
            plt.title("Prediction")
            plt.axis("off")

            plt.tight_layout()
            plt.show()

mean_psnr = np.mean(psnr_vals)
mean_ssim = np.mean(ssim_vals)
print(f"[*] Evaluation --- Mean PSNR: {mean_psnr:.3f}")
print(f"[*] Evaluation --- Mean SSIM: {mean_ssim:.3f}")

In [ ]:
# Save 5 test views in validation set
# You can find these images on the workspace of Colab
# !!! Important: Please save them to your local computer before the Colab runtime is deleted !!!

def save_numpy_to_png(pred_npy, output_path):
    pred_clipped = np.clip(pred_npy, 0.0, 1.0)
    
    # 2. Scale from [0, 1] to [0, 255] and convert to 8-bit unsigned integer
    pred_uint8 = (pred_clipped * 255.0).astype(np.uint8)

    img = Image.fromarray(pred_uint8)
    img.save(output_path)
    print(f"Saved image successfully to {output_path}")

test_numbers = [0, 20, 40, 60, 80, 100, 120, 140, 160, 180]

for i, test_data in enumerate(tqdm(test_loader, desc="Running Evaluation")):

    gt_img, camera, gt_mask = test_data
    gt_img = gt_img[0].numpy()
    camera = camera[0].cuda() if device == "cuda" else camera[0]
    if gt_mask is not None:
        gt_mask = gt_mask[0].to(device)

    with torch.no_grad():

        if i in test_numbers:

            if gt_mask is not None:
                gt_mask = gt_mask[0].to(device)
            pred_img, _, _ = renderer.render(camera=camera, per_splat=-1, img_size=(gt_img.shape[1], gt_img.shape[0]),bg_colour=(0,0,0))

            pred_npy = pred_img.detach().cpu().numpy()
            save_numpy_to_png(pred_npy, f'./val_{i}.png')